In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when
from pyspark.sql.types import TimestampType


In [0]:
dbutils.widgets.text("catalog_name", "")
catalog_name = dbutils.widgets.get("catalog_name")
dbutils.widgets.text("schema_bronze_name", "")
schema_bronze_name = dbutils.widgets.get("schema_bronze_name")
dbutils.widgets.text("schema_silver_name", "")
schema_silver_name = dbutils.widgets.get("schema_silver_name")

In [0]:
bronze_table = f"{catalog_name}.{schema_bronze_name}.crime_evh_bronze"
checkpoint_silver_loc = f"/Volumes/{catalog_name}/{schema_silver_name}/silver/checkpoints/"
target_silver_scd1 = f"{catalog_name}.{schema_silver_name}.crime_silver_scd1"
target_silver_scd2 = f"{catalog_name}.{schema_silver_name}.crime_silver_scd2"

In [0]:
bronze_stream_df = (
    spark.readStream.format("delta")     
    .table(bronze_table)  
)

In [0]:
silver_table = (bronze_stream_df.select("value.*", bronze_stream_df.timestamp.alias("bronze_ingestion_time"))\
    .withColumn("silver_ingestion_time", F.current_timestamp())    
)

#### empty strings into nulls

In [0]:
silver_cols = silver_table.columns

def blank_as_null(x):    
    return when(col(x) != "", col(x)).otherwise(None)

#### Not changing for timestamps because we have null values

In [0]:
updates = {f"{col_name}": blank_as_null(col_name) for col_name in silver_cols if not isinstance(silver_table.schema[col_name].dataType, TimestampType)}   
silver_table = silver_table.withColumns(updates)

In [0]:
columns_to_cast = {
    "suspect_age": "double",
    "latitude": "double",
    "longitude": "double",
    "victim_age": "double",
    "num_arrests": "double",
    "property_loss_usd": "double"
}

for col_name, target_type in columns_to_cast.items():
    if col_name in silver_cols:
        silver_table = silver_table.withColumn(col_name, F.col(col_name).try_cast(target_type))

In [0]:
query = ( silver_table.writeStream \
  .format("delta") \
  .outputMode("append") \
  .option("checkpointLocation", checkpoint_silver_loc)  
  .trigger(availableNow=True)
  .toTable(target_silver_scd1)     
)

In [0]:
query = spark.sql(f"SELECT * FROM {target_silver_scd1} ORDER BY suspect_id")
display(query)

### SCD1

In [0]:
silver_table.createOrReplaceTempView("silver_table")

spark.sql(
"""
MERGE INTO target_silver_scd1 TARGET
USING silver_table SOURCE
ON TARGET.PersonId = SOURCE.PersonId
WHEN MATCHED THEN
  UPDATE SET
    TARGET.FirstName = SOURCE.FirstName,
    TARGET.LastName = SOURCE.LastName,
    TARGET.Country = SOURCE.Country
WHEN NOT MATCHED THEN
  INSERT
  (
    PersonId,
    FirstName,
    LastName,
    Country
  )
  VALUES
  (
    SOURCE.PersonId,
    SOURCE.FirstName,
    SOURCE.LastName,
    SOURCE.Country
  )
"""
)